# Generación de Texto con RNN LSTM (Long Short-Term Memory)

Este cuaderno tiene como objetivo reforzar el aprendizaje de redes neuronales recurrentes (RNN) y LSTMs utilizando Keras con TensorFlow para la tarea de generación de texto. A través de este ejercicio, los estudiantes aprenderán cómo se puede generar texto de manera automática utilizando un modelo LSTM entrenado con un conjunto de datos de titulares de noticias.

### Generación de Titulares de Noticias

En este cuanderno, utilizaremos el conjunto de datos de [Comentarios y Titulares del New York Times](https://www.kaggle.com/aashita/nyt-comments) para entrenar un modelo de lenguaje para generación de texto, que pueda ser utilizado para generar titulares de noticias.

## 1. Importación de Bibliotecas

Primero, importamos todas las bibliotecas necesarias para crear y entrenar nuestro modelo.

* keras.preprocessing.sequence: Usado para preprocesar secuencias, como la función pad_sequences.
* keras.layers: Incluye las capas que se usarán en la arquitectura del modelo, como LSTM y Dense.
* keras.callbacks: Permite implementar controladores de entrenamiento como EarlyStopping.
* tensorflow: Usamos esto para configurar las semillas de aleatorización para reproducibilidad.
* pandas y numpy: Para la manipulación de datos.*
warnings: Para ignorar advertencias irrelevantes en el cuaderno.

In [ ]:
!pip install keras tensorflow

* **from tensorflow.keras.preprocessing.sequence import pad_sequences:** Importa pad_sequences desde tensorflow.keras.
* **from tensorflow.keras.layers import Embedding**, LSTM, Dense, Dropout: Cambia la importación de las capas LSTM y otras desde keras a tensorflow.keras.layers.
* **from tensorflow.keras.preprocessing.text import Tokenizer**: Similarmente, cambia la importación de Tokenizer de Keras a tensorflow.keras.preprocessing.text.
* **from tensorflow.keras.callbacks import EarlyStopping**: Cambia la importación de EarlyStopping.
* **import tensorflow.keras.utils as ku**: Actualiza la importación de las utilidades de Keras.
* **from tensorflow import random as tf_random**: La configuración de semillas también debe importarse desde tensorflow.random si se usa TensorFlow 2.x.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
import tensorflow.keras.utils as ku
from tensorflow import random as tf_random
from numpy.random import seed
import pandas as pd
import numpy as np
import string, os
import warnings

In [ ]:
# Para hacer las ejecuciones reproducibles
tf_random.set_seed(2)
seed(1)

# Para evitar advertencias innecesarias
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)


#2. Web Scraping de El Tiempo

Si no hay una API oficial disponible, una alternativa es hacer web scraping del sitio web de El Tiempo. Esto te permitiría extraer los titulares y contenido de las noticias directamente desde las páginas web de El Tiempo.

Nota: El scraping de sitios web puede estar sujeto a restricciones legales y de uso de los datos. Siempre sería importante revisar los Términos de servicio del sitio web antes de realizar scraping.

Vamos a usar bibliotecas como BeautifulSoup y requests en Python para obtener el contenido de las páginas.

In [ ]:
import requests
from bs4 import BeautifulSoup

# Realiza una solicitud GET a la página de El Tiempo
url = "https://www.eltiempo.com/"
response = requests.get(url)

# Parsear el contenido HTML
soup = BeautifulSoup(response.text, 'html.parser')

# Extraer todos los textos relevantes, por ejemplo: encabezados h1, h2, h3 y párrafos p
headlines_h1 = soup.find_all('h1')  # Titulares importantes
headlines_h2 = soup.find_all('h2')  # Subtitulares
headlines_h3 = soup.find_all('h3')  # Subcategorías o secciones
paragraphs = soup.find_all('p')  # Párrafos de contenido

# Crear una lista para almacenar todos los textos
headlines = []

# Añadir los textos de los encabezados h1, h2, h3 y los párrafos a la lista headlines
for headline in headlines_h1 + headlines_h2 + headlines_h3 + paragraphs:
    text = headline.get_text().strip()  # Eliminar espacios y tabs al principio y final
    if text and len(text) > 15:  # Ignorar líneas vacías y de longitud menor a 15 caracteres
        headlines.append(text)

# Imprimir todos los textos limpios
for text in headlines:
    print(text)


Últimas Noticias de Colombia y el Mundo
Hola, bienvenido
Estos son los impuestos que vendrían con el nuevo intento de reforma tributaria
El otro hombre que estuvo en la entrega del jugoso aporte de 'Papá Pitufo' a la campaña Petro
Robo masivo en conjunto residencial de Bogotá: manada de ladrones amordazó a las víctimas
Las piezas ocultas de infiltración de 'Papá Pitufo' que desataron tormenta en Palacio y Policía
Las 2 pistas del crimen de cantante Zair Guette; estaba en lista de objetivos de banda criminal
América vs. Santa Fe, minuto a minuto en la Liga Betplay; Juanfer Quintero, titular
REPORTAJES MULTIMEDIA
TU NEGOCIO A UN CLIC
NUESTROS PORTALES
Así queda la cúpula de la Policía tras la llegada del nuevo director, general Carlos Triana
'Lo sufría': DT del León de México confesó tormentosa realidad que vivía con James Rodríguez
Las razones detrás de la llegada de depredadores sexuales extranjeros a Colombia: ¿qué hacer?
Shakira y su talento camaleónico para conquistar tres generacio

In [ ]:
# Función para eliminar frases en mayúsculas
def remove_uppercase_lines(headlines):
    headlines = headlines[1:]  # Removing the first headline, which was causing the issue
    cleaned_headlines = []

    for headline in headlines:
        # Check if the headline is a string and is uppercase, skip if so
        if isinstance(headline, str) and headline.isupper():
            continue
        cleaned_headlines.append(headline)

    return cleaned_headlines

# Limpiar los titulares
cleaned_headlines = remove_uppercase_lines(headlines)

# Imprimir los titulares filtrados
for headline in cleaned_headlines:
    if isinstance(headline, str):  # Check if the headline is a string
        print(headline)
    else:
        print(headline.get_text()) # If not a string, assume it's a Tag object

Hola, bienvenido
Estos son los impuestos que vendrían con el nuevo intento de reforma tributaria
El otro hombre que estuvo en la entrega del jugoso aporte de 'Papá Pitufo' a la campaña Petro
Robo masivo en conjunto residencial de Bogotá: manada de ladrones amordazó a las víctimas
Las piezas ocultas de infiltración de 'Papá Pitufo' que desataron tormenta en Palacio y Policía
Las 2 pistas del crimen de cantante Zair Guette; estaba en lista de objetivos de banda criminal
América vs. Santa Fe, minuto a minuto en la Liga Betplay; Juanfer Quintero, titular
Así queda la cúpula de la Policía tras la llegada del nuevo director, general Carlos Triana
'Lo sufría': DT del León de México confesó tormentosa realidad que vivía con James Rodríguez
Las razones detrás de la llegada de depredadores sexuales extranjeros a Colombia: ¿qué hacer?
Shakira y su talento camaleónico para conquistar tres generaciones: ¿cuál es su secreto?
Luigi Mangione habló por primera vez desde cárcel tras crimen de Brian Thom

## 3. Preparación del dataset

### 3.1 Limpiar el dataset

Limpiamos el conjunto de datos eliminando puntuaciones y convirtiendo todas las palabras a minúsculas.


In [ ]:
def clean_text(txt):
    # Check if txt is a BeautifulSoup Tag object
    if hasattr(txt, 'get_text'):
        txt = "".join(v for v in txt.get_text() if v not in string.punctuation).lower()
    # If it's a string, just convert it to lowercase
    elif isinstance(txt, str):
        txt = "".join(v for v in txt if v not in string.punctuation).lower()
    # If it's neither, handle it as needed (e.g., raise an exception)
    else:
        raise TypeError("Input must be a string or a BeautifulSoup Tag object")
    # Remove the line that causes the issue:
    # txt = txt.encode("utf8").decode("ascii",'ignore')
    return txt

corpus = [clean_text(x) for x in headlines]
print(corpus[:10])  # Muestra los primeros 10 titulares limpiados

['últimas noticias de colombia y el mundo', 'hola bienvenido', 'estos son los impuestos que vendrían con el nuevo intento de reforma tributaria', 'el otro hombre que estuvo en la entrega del jugoso aporte de papá pitufo a la campaña petro', 'robo masivo en conjunto residencial de bogotá manada de ladrones amordazó a las víctimas', 'las piezas ocultas de infiltración de papá pitufo que desataron tormenta en palacio y policía', 'las 2 pistas del crimen de cantante zair guette estaba en lista de objetivos de banda criminal', 'américa vs santa fe minuto a minuto en la liga betplay juanfer quintero titular', 'reportajes multimedia', 'tu negocio a un clic']


### 3.2 Generar secuencias de tokens N-gram

Generaremos secuencias de N-gramas para entrenar nuestro modelo.


In [ ]:
tokenizer = Tokenizer()

def get_sequence_of_tokens(corpus):
    tokenizer.fit_on_texts(corpus)
    total_words = len(tokenizer.word_index) + 1

    input_sequences = []
    for line in corpus:
        token_list = tokenizer.texts_to_sequences([line])[0]
        for i in range(1, len(token_list)):
            n_gram_sequence = token_list[:i+1]
            input_sequences.append(n_gram_sequence)
    return input_sequences, total_words

inp_sequences, total_words = get_sequence_of_tokens(corpus)
print(inp_sequences[:10])  # Muestra las primeras 10 secuencias N-gram


[[250, 105], [250, 105, 1], [250, 105, 1, 12], [250, 105, 1, 12, 5], [250, 105, 1, 12, 5, 4], [250, 105, 1, 12, 5, 4, 19], [106, 107], [54, 55], [54, 55, 10], [54, 55, 10, 251]]


### 3.3 Padding de las secuencias y creación de variables para predicción y etiqueta

Ahora aplicamos padding a las secuencias para que todas tengan la misma longitud. Además, se separan los datos en predictores y etiquetas.


In [ ]:
def generate_padded_sequences(input_sequences):
    max_sequence_len = max([len(x) for x in input_sequences])
    input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

    predictors, label = input_sequences[:,:-1],input_sequences[:,-1]
    label = ku.to_categorical(label, num_classes=total_words)
    return predictors, label, max_sequence_len

predictors, label, max_sequence_len = generate_padded_sequences(inp_sequences)
print(f'Longitud máxima de las secuencias: {max_sequence_len}')


Longitud máxima de las secuencias: 55


In [ ]:
predictors

array([[   0,    0,    0, ...,    0,    0,  250],
       [   0,    0,    0, ...,    0,  250,  105],
       [   0,    0,    0, ...,  250,  105,    1],
       ...,
       [   0,    0,    0, ...,    0, 1026,    1],
       [   0,    0,    0, ..., 1026,    1, 1027],
       [   0,    0,    0, ...,    1, 1027,    5]], dtype=int32)

In [ ]:
label

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]])

## 4. Crear el modelo LSTM para la generación de texto

Ahora vamos a crear un modelo LSTM con 3 capas:

1. **Capa de Entrada**: Capa de embedding.
2. **Capa LSTM**: Capa LSTM con 100 unidades.
3. **Capa de Salida**: Capa densa para predecir la siguiente palabra en la secuencia.


In [ ]:
def create_model(max_sequence_len, total_words):
    input_len = max_sequence_len - 1
    model = Sequential()

    # Capa de entrada - Embedding
    model.add(Embedding(total_words, 10, input_length=input_len))

    # Capa LSTM
    model.add(LSTM(100))
    model.add(Dropout(0.1))  # Capa de Dropout para regularización

    # Capa de salida
    model.add(Dense(total_words, activation='softmax'))

    # Compilación del modelo
    model.compile(loss='categorical_crossentropy', optimizer='adam')

    return model

model = create_model(max_sequence_len, total_words)
model.summary()  # Mostrar resumen del modelo


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 5. Entrenar el modelo

Entrenamos el modelo utilizando las secuencias generadas como predictores y etiquetas.


In [ ]:
model.fit(predictors, label, epochs=100, verbose=5)


Epoch 1/100
Epoch 2/100
Epoch 3/100
Epoch 4/100
Epoch 5/100
Epoch 6/100
Epoch 7/100
Epoch 8/100
Epoch 9/100
Epoch 10/100
Epoch 11/100
Epoch 12/100
Epoch 13/100
Epoch 14/100
Epoch 15/100
Epoch 16/100
Epoch 17/100
Epoch 18/100
Epoch 19/100
Epoch 20/100
Epoch 21/100
Epoch 22/100
Epoch 23/100
Epoch 24/100
Epoch 25/100
Epoch 26/100
Epoch 27/100
Epoch 28/100
Epoch 29/100
Epoch 30/100
Epoch 31/100
Epoch 32/100
Epoch 33/100
Epoch 34/100
Epoch 35/100
Epoch 36/100
Epoch 37/100
Epoch 38/100
Epoch 39/100
Epoch 40/100
Epoch 41/100
Epoch 42/100
Epoch 43/100
Epoch 44/100
Epoch 45/100
Epoch 46/100
Epoch 47/100
Epoch 48/100
Epoch 49/100
Epoch 50/100
Epoch 51/100
Epoch 52/100
Epoch 53/100
Epoch 54/100
Epoch 55/100
Epoch 56/100
Epoch 57/100
Epoch 58/100
Epoch 59/100
Epoch 60/100
Epoch 61/100
Epoch 62/100
Epoch 63/100
Epoch 64/100
Epoch 65/100
Epoch 66/100
Epoch 67/100
Epoch 68/100
Epoch 69/100
Epoch 70/100
Epoch 71/100
Epoch 72/100
Epoch 73/100
Epoch 74/100
Epoch 75/100
Epoch 76/100
Epoch 77/100
Epoch 78

## 6. Generación de texto

Ahora que el modelo está entrenado, podemos usarlo para generar texto basado en una semilla (texto inicial) y predecir las siguientes palabras en la secuencia.

### **Función generate_text**

Esta función es la encargada de generar nuevo texto utilizando el modelo LSTM que has entrenado. Recibe cuatro argumentos:

**seed_text:** Es el texto inicial que se usa como semilla para la generación.
next_words: Indica el número de palabras que se desean generar.
model: Es el modelo LSTM entrenado.
max_sequence_len: La longitud máxima de las secuencias de entrada del modelo.
Dentro de la función:

**Bucle for:** Este bucle se ejecuta next_words veces, generando una palabra en cada iteración.

**token_list = tokenizer.texts_to_sequences([seed_text])[0]:** Se utiliza el tokenizador (tokenizer) para convertir el seed_text en una secuencia de tokens (números que representan cada palabra).

**token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre'):** Se aplica padding (relleno) a la secuencia de tokens para que tenga la longitud esperada por el modelo (max_sequence_len - 1). El padding se hace al principio (padding='pre').

**predicted = np.argmax(model.predict(token_list), axis=-1):** Se utiliza el modelo (model) para predecir la siguiente palabra. model.predict devuelve una distribución de probabilidad sobre todas las palabras posibles, y np.argmax selecciona el índice (que corresponde a una palabra) con la mayor probabilidad.

**Bucle for:**** Este bucle busca la palabra que corresponde al índice predicho (predicted) en el vocabulario del tokenizador (tokenizer.word_index).

seed_text += " " + output_word:** Se agrega la palabra predicha (output_word) al seed_text, que se va actualizando en cada iteración del bucle principal.

**return seed_text.title():** Finalmente, se devuelve el texto generado, con la primera letra de cada palabra en mayúscula (title()).**

In [ ]:
def generate_text(seed_text, next_words, model, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text.title()


In [ ]:
headlines

['Últimas Noticias de Colombia y el Mundo',
 'Hola, bienvenido',
 'Estos son los impuestos que vendrían con el nuevo intento de reforma tributaria',
 "El otro hombre que estuvo en la entrega del jugoso aporte de 'Papá Pitufo' a la campaña Petro",
 'Robo masivo en conjunto residencial de Bogotá: manada de ladrones amordazó a las víctimas',
 "Las piezas ocultas de infiltración de 'Papá Pitufo' que desataron tormenta en Palacio y Policía",
 'Las 2 pistas del crimen de cantante Zair Guette; estaba en lista de objetivos de banda criminal',
 'América vs. Santa Fe, minuto a minuto en la Liga Betplay; Juanfer Quintero, titular',
 'REPORTAJES MULTIMEDIA',
 'TU NEGOCIO A UN CLIC',
 'NUESTROS PORTALES',
 'Así queda la cúpula de la Policía tras la llegada del nuevo director, general Carlos Triana',
 "'Lo sufría': DT del León de México confesó tormentosa realidad que vivía con James Rodríguez",
 'Las razones detrás de la llegada de depredadores sexuales extranjeros a Colombia: ¿qué hacer?',
 'Shaki

In [ ]:
palabras=input("Ingrese una o dos palabras iniciales: ") # Palabras relacionadas con los titulares
next_words=int(input("Ingrese el número de palabras que desea generar: ")) # Número de palabras a generar
print(generate_text(palabras, 5, model, max_sequence_len))


Ingrese una o dos palabras iniciales: Valentina
Ingrese el número de palabras que desea generar: 2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Valentina Bajonero Vásquez Elige Entre Los


## 8. Ideas de mejora

Para mejorar los resultados generados, puedes probar los siguientes enfoques:

- Añadir más datos de entrenamiento.
- Ajustar la arquitectura del modelo (número de unidades LSTM, capas adicionales, etc.).
- Probar con diferentes hiperparámetros (tasa de aprendizaje, tamaño de la capa, etc.).
